# 11 | LangGraph checkpoint：状态快照到底是什么

Checkpoint 是 LangGraph 为 State 保存的状态快照。它解决的问题是：图运行到某一步以后，如何保留当时的状态，后续还能继续读取、恢复或追踪。

Checkpoint 可以先理解成：LangGraph 在一段会话里，为图的 State 留下的存档点。

它不是一个单独的“聊天记录变量”，也不是只能给聊天机器人用的记忆功能。只要你的图有 State，checkpoint 保存的就是这个 State 在运行过程中的快照。

## 一、先抓住三个关键词

理解 checkpoint，先抓住三个关键词就够了：

| 关键词 | 作用 |
| --- | --- |
| `State` | 图当前知道什么 |
| `thread_id` | 这份状态属于哪一段会话 |
| `checkpointer` | 把状态保存到哪里 |

可以把它想成读书摘记：

```text
thread_id = reading-notes-001
  checkpoint 1: 已记录第一条摘记
  checkpoint 2: 已记录第二条摘记
  checkpoint 3: 已生成新的整理结果
```

`thread_id` 决定读哪一本“摘记本”，checkpoint 记录这本摘记本在不同时间点的状态。


In [ ]:
from importlib.metadata import version

print("langgraph", version("langgraph"))

## 二、一个读书摘记 State

下面的 State 很小，只有两个字段：

- `notes`：用户输入的原始摘记；
- `digests`：图整理出来的简短摘要。

两个字段都使用 `operator.add` 作为 reducer。也就是说，每次节点返回新列表时，LangGraph 会把它追加到旧列表后面。


In [ ]:
import operator
from typing import Annotated, TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, StateGraph


class ReadingState(TypedDict):
    notes: Annotated[list[str], operator.add]
    digests: Annotated[list[str], operator.add]


def make_digest(state: ReadingState) -> dict[str, list[str]]:
    latest_note = state["notes"][-1]
    digest_no = len(state.get("digests", [])) + 1
    digest = f"摘记 {digest_no}: {latest_note[:28]}"
    return {"digests": [digest]}


builder = StateGraph(ReadingState)
builder.add_node("make_digest", make_digest)
builder.add_edge(START, "make_digest")
builder.add_edge("make_digest", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

## 三、同一个 `thread_id` 会接上同一份 State

启用 checkpoint 后，调用图时要传入 `thread_id`。

第一次运行后，LangGraph 会保存这段会话的 State。第二次继续用同一个 `thread_id`，新摘记会被追加到旧 State 里。


In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "reading-notes-001"}}

first_result = graph.invoke(
    {
        "notes": ["《刻意练习》强调，有反馈的训练比单纯重复更重要。"],
        "digests": [],
    },
    config=config,
)

second_result = graph.invoke(
    {
        "notes": ["《深度工作》提醒我，把完整时间块留给高认知任务。"],
        "digests": [],
    },
    config=config,
)

print("notes:")
for note in second_result["notes"]:
    print("-", note)

print("\ndigests:")
for digest in second_result["digests"]:
    print("-", digest)

## 四、checkpoint 保存的是完整 State 快照

`get_state(config)` 会读取这个 `thread_id` 当前最新的 checkpoint。

注意这里读到的不是某个单独字段，而是完整的 State：`notes` 和 `digests` 都在里面。


In [ ]:
snapshot = graph.get_state(config)

checkpoint_config = snapshot.config.get("configurable", {})

print("checkpoint_id:", checkpoint_config.get("checkpoint_id"))
print("next:", snapshot.next)
print("values:")
for key, value in snapshot.values.items():
    print(f"- {key}: {value}")

## 五、要点

记住三点：

- checkpoint 保存的是图的 State 快照；
- `thread_id` 用来区分不同会话的状态历史；
- State 字段如何合并，仍然由 reducer 决定。

所以下一篇再看多轮对话时，重点就不是“LangGraph 有没有记忆”，而是：同一个 `thread_id` 下，新的输入如何和旧 State 合并。
